# RedFlag Lens: how we trained the models, explained step by step

**Navnit Naman (230085) · Kanhaiya Kumar (230062)**

This notebook walks through the whole training story in the order we did it, with comments on every step,
so we can explain each decision in the viva.

**The story in one line:** we taught a computer to read an annual report and point at the sentences that are
warning signs. We started with the simplest idea (keyword search), saw it fail, and improved step by step until
we reached a model that finds 92% of real risks.

| Step | What we try | Result (F1 on 600 test sentences) |
|---|---|---|
| 1 | Keyword search, like Ctrl+F | 0.23 |
| 2 | TF-IDF + logistic regression (counting important words) | 0.60 |
| 3 | FinBERT, fine-tuned (understands meaning) | 0.77 |
| 4 | DeBERTa-v3-large + extra examples, 3 copies averaged | **0.86** |
| 5 | Cascade: FinBERT screens, DeBERTa re-checks (fast on a laptop) | 0.83 |
| 6 | Severity rule + whole-report score | ROC-AUC 0.94 on 8 real reports |

**How to run it:** *Run All*. With the default settings it takes about 2 minutes, because it **loads the models we
already trained** instead of training them again. To retrain FinBERT yourself, set `RUN_TRAINING = True` in the
next cell (about 10–20 minutes on a laptop). DeBERTa-v3-large needs a GPU to train (we used an RTX 4090 / L40S on RunPod).

The real project code lives in `src/`. This notebook uses the same functions and the same settings, so the numbers
here are the numbers in our report and slides.

## 1. Setup

We import the libraries and tell Python where our project folders are.

In [1]:
import json, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")              # hide library warnings so the output stays readable

# Find the project folder (works whether the notebook is opened from notebooks/ or from the project root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))                  # lets us import our app code:  src.app.pipeline
sys.path.insert(0, str(ROOT / "src/models"))   # lets us import our model code: baselines, rules

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

# False = load the models we already trained (fast).  True = train FinBERT again from scratch (slow).
RUN_TRAINING = False

pd.set_option("display.max_colwidth", 150)
print("Project folder:", ROOT.name)

Project folder: RedFlagLens


## 2. The data: labelled sentences from real annual reports

We downloaded **26 annual reports** from the BSE (10 companies that later failed, 16 healthy ones), extracted about
**97,000 sentences**, and labelled a sample of them. Each labelled sentence has:

- `is_risk`: 1 if it is a warning sign, 0 if it is normal business text
- `category`: the type of risk, one of **Financial, Legal, Regulatory, Operational, Business**
- `section`: which part of the report it came from (for example the auditor's report)

**Important design choice: we split by company, not by sentence.** About 28% of a report's text is copied from the
previous year. If the same company were in both training and testing, the model could simply memorise those sentences,
and the test score would be fake-good. So each company goes into only one group.

In [2]:
DATA = ROOT / "data/labels/v1.1"   # v1.1 = the final version of our labels (600-sentence test set)

def load(split):
    # Read one split (train / val / test) of labelled sentences
    d = pd.read_json(DATA / f"{split}.jsonl", lines=True)
    d["is_risk"] = d.is_risk.astype(int)                    # make sure the label is 0/1
    d["category"] = d.category.fillna("None").astype(str)   # non-risk sentences have no category -> "None"
    # The model reads the section name in front of the sentence, e.g. "[auditor_report] The Company has defaulted..."
    # because WHERE a sentence appears is a useful clue (an auditor's remark matters more than a marketing line).
    d["input"] = "[" + d.section.astype(str) + "] " + d.text.astype(str)
    return d

train, val, test = load("train"), load("val"), load("test")

summary = pd.DataFrame({
    "split": ["train (teach the model)", "validation (choose settings)", "test (final exam)"],
    "sentences": [len(train), len(val), len(test)],
    "risk sentences": [train.is_risk.sum(), val.is_risk.sum(), test.is_risk.sum()],
    "companies": [", ".join(sorted(d.company.unique())) if len(d.company.unique()) <= 5 else f"{d.company.nunique()} companies"
                  for d in (train, val, test)],
})
summary["% risk"] = (100 * summary["risk sentences"] / summary["sentences"]).round(1)
summary

,split,sentences,risk sentences,companies,% risk
0,train (teach the model),1590,211,16 companies,13.3
1,validation (choose settings),260,30,"NTPC, RCOM, TITAN",11.5
2,test (final exam),600,37,"JETAIRWAYS, MARUTI, SUNPHARMA, TCS, YESBANK",6.2


**Leakage check.** The cell below proves that no company appears in two groups, and that Gensol (our real-life check,
collapsed in 2025) is in none of them.

In [3]:
tr_c, va_c, te_c = set(train.company), set(val.company), set(test.company)

# If any of these fails, Python stops with an error, so a clean run is the proof.
assert not (tr_c & va_c), "a company is in both train and validation"
assert not (tr_c & te_c), "a company is in both train and test"
assert not (va_c & te_c), "a company is in both validation and test"
assert "GENSOL" not in (tr_c | va_c | te_c), "Gensol must be kept out of everything"

print("OK: every company is in exactly one group, and Gensol is in none of them.")
print("Validation companies:", sorted(va_c))
print("Test companies:      ", sorted(te_c))

OK: every company is in exactly one group, and Gensol is in none of them.
Validation companies: ['NTPC', 'RCOM', 'TITAN']
Test companies:       ['JETAIRWAYS', 'MARUTI', 'SUNPHARMA', 'TCS', 'YESBANK']


**What the labels look like.** One real example of each risk type from the training set, plus two normal sentences.
(We only pick whole sentences of a readable length here, so the examples are easy to read.)

In [4]:
readable = train[train.text.str.len().between(70, 230) & train.text.str[0].str.isupper()]   # whole sentences only
examples = pd.concat(
    [readable[(readable.is_risk == 1) & (readable.category == c)].sample(1, random_state=3)
     for c in ["Financial", "Legal", "Regulatory", "Operational", "Business"]]     # one warning sign per type
    + [readable[readable.is_risk == 0].sample(2, random_state=3)]                  # two normal sentences
)
examples[["company", "is_risk", "category", "text"]]

,company,is_risk,category,text
1317,DHFL,1,Financial,This coupled with multiple downgrades by rating agencies created significant pressure on your Company to maintain a stable liability profile and r...
700,DRREDDY,1,Legal,"The Company's interlocutory appeal of the said denial, was also denied."
1241,LT,1,Regulatory,SEBI vide its letter SEBI/HO/CFD/DCR1/OW/P/2019/2008/1 dated 18th January 2019 advised the Company not to proceed with the aforesaid proposal of b...
853,DHFL,1,Operational,"Consequent to this, there have been instances where cheques received from the borrowers particularly from project and mortgage loan customers, wer..."
1150,HAVELLS,1,Business,Being a Fast-Moving Electrical Goods (FMEG) company we are operating in product portfolios which are characterized by rapid changes in consumer pr...
211,DRREDDY,0,None,Our opinion on the consolidated Ind AS financial statements does not cover the other information and we do not and will not express any form of as...
1022,COFFEEDAY,0,None,The notes referred to above form an integral part of the standalone financial statements


In [5]:
# How many risk sentences of each type do we have for training?
# Financial is most common; Business is rare, which is one reason naming the type is harder than finding the risk.
train[train.is_risk == 1].category.value_counts().rename("risk sentences in train").to_frame()

,risk sentences in train
category,
Financial,96
Regulatory,48
Legal,43
Operational,14
Business,10


## 3. How we measure a model: precision, recall and F1

- **Precision**: of all the sentences the model flagged, how many were really risks? (Are its alarms trustworthy?)
- **Recall**: of all the real risks, how many did the model find? (Does it miss things?)
- **F1**: one number that balances both. 1.0 is perfect.

For a bank, **recall matters most**: missing a real default is much worse than reading a few extra sentences.

A tiny example with 5 sentences, 2 of them real risks:

In [6]:
y_true = [1, 1, 0, 0, 0]   # sentences 1 and 2 are real risks
y_pred = [1, 0, 1, 0, 0]   # the model flagged sentences 1 and 3

print("precision =", precision_score(y_true, y_pred), " -> 1 of its 2 flags is real")
print("recall    =", recall_score(y_true, y_pred), " -> it found 1 of the 2 real risks")
print("F1        =", f1_score(y_true, y_pred))

results = []   # we collect every model's score here and compare them at the end

def score(name, y, pred):
    # Compute the three numbers for one model on the test set and remember them
    row = {"model": name,
           "precision": round(precision_score(y, pred, zero_division=0), 2),
           "recall": round(recall_score(y, pred, zero_division=0), 2),
           "F1": round(f1_score(y, pred, zero_division=0), 2)}
    results.append(row)
    return row

precision = 0.5  -> 1 of its 2 flags is real
recall    = 0.5  -> it found 1 of the 2 real risks
F1        = 0.5


## 4. Step 1: keyword search (what an analyst does with Ctrl+F)

The simplest idea: flag any sentence that contains words like *default, going concern, NCLT, fraud, penalty*.
No training at all. This is our "floor": every real model must beat it.

In [7]:
from baselines import RISK_KW   # the same keyword list used in our project code

print("Some of the keywords:", RISK_KW.pattern[:200], "...")

# 1 if the sentence contains any keyword, else 0
pred_kw = test.text.str.contains(RISK_KW).astype(int).values
score("1. Keyword search (Ctrl+F)", test.is_risk, pred_kw)

Some of the keywords: going concern|default|overdue|penalt|show cause|litigation|lawsuit|suit filed|arbitration|claim|NCLT|NCLAT|insolvency|IBC|winding up|fraud|wilful|invok|downgrad|impair|write.?off|provision for|qualifi ...


{'model': '1. Keyword search (Ctrl+F)',
 'precision': 0.13,
 'recall': 0.73,
 'F1': 0.23}

**Why it fails:** recall is decent (it finds most risks) but precision is terrible (0.13): 87% of its flags are false
alarms. Words like *default* or *penalty* also appear in harmless sentences such as *"there has been **no default** in
repayment"* or standard legal templates. Keywords can't understand meaning.

## 5. Step 2: TF-IDF + logistic regression (the classic machine-learning baseline)

- **TF-IDF** turns each sentence into numbers: a score for every word (and short word pairs and letter groups).
  Words that are rare across reports get higher scores, because they carry more information.
- **Logistic regression** then learns, from our labelled examples, which words push a sentence towards "risk".

This is real learning, but it still only counts words; it doesn't understand sentences.

In [8]:
from sklearn.linear_model import LogisticRegression
from baselines import fit_vec, make_vec, tf   # our TF-IDF settings: word 1-2 grams + character 3-5 grams

t0 = time.time()
vecs = make_vec()                                 # create the two TF-IDF converters
X_train = fit_vec(vecs, train.input)              # learn the vocabulary from TRAINING sentences only
clf = LogisticRegression(
    C=4,                       # how strongly to fit the data (chosen on validation)
    class_weight="balanced",   # risks are only ~13% of training data, so we give them more weight
    max_iter=2000, random_state=42,
).fit(X_train, train.is_risk)

test_scores = clf.predict_proba(tf(vecs, test.input))[:, 1]   # risk probability (0 to 1) for every test sentence
pred_tfidf = (test_scores >= 0.5).astype(int)                  # flag it if the probability is at least 0.5
print(f"trained in {time.time() - t0:.1f} seconds")
score("2. TF-IDF + logistic regression", test.is_risk, pred_tfidf)

trained in 6.2 seconds


{'model': '2. TF-IDF + logistic regression',
 'precision': 0.52,
 'recall': 0.7,
 'F1': 0.6}

**Better (0.60), but not enough.** It still mixes up harmless and risky sentences that share the same words.
To go further we need a model that reads the **whole sentence** and understands it.

## 6. Step 3: fine-tuning FinBERT, a model that understands financial language

**What is FinBERT?** A BERT language model that has already read huge amounts of financial text, so it already
"knows" English and finance. We don't build it from zero; we **fine-tune** it: we show it our 1,590 labelled
sentences and adjust it slightly so it becomes an expert at our exact task.

**Our trick: one model, six answers.** Instead of separate models for "is it a risk?" and "what type?", the model
picks one of six labels: `None` (no risk) or one of the 5 risk types. Then:

- risk score = 1 − P(None)   (how sure it is that the sentence is *some* kind of risk)
- risk type = the most likely of the 5 risk labels

**The training recipe** (the same as `src/models/finbert.py`):

| Setting | Value | Why, in simple words |
|---|---|---|
| Epochs | 4 | read the training set 4 times |
| Learning rate | 3e-5 | small steps, so we don't destroy what FinBERT already knows |
| Batch size | 16 | learn from 16 sentences at a time |
| Max length | 128 tokens | long enough for almost every sentence |
| Class weights | softened (square root) | risks are rare; give them more weight, but not so much that it panics |
| Warm-up | first 10% of steps | start gently, then go full speed, then slow down |
| Gradient clipping | 1.0 | stops one weird batch from making a huge jump |

In [9]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

LABELS = ["None", "Financial", "Legal", "Regulatory", "Operational", "Business"]   # the six answers
L2I = {label: i for i, label in enumerate(LABELS)}                                 # label name -> number
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


def to_label(df):
    # Turn (is_risk, category) into one of the six numbers: 0 = None, 1..5 = risk types
    return [L2I[c if r else "None"] for c, r in zip(df.category, df.is_risk)]


def batches(tok, texts, labels=None, bs=16, shuffle=False, seed=0):
    # Cut the sentences into small groups, and turn the words into token numbers the model understands
    dl = DataLoader(list(range(len(texts))), batch_size=bs, shuffle=shuffle,
                    generator=torch.Generator().manual_seed(seed))
    for b in dl:
        enc = tok([texts[i] for i in b], truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEV)
        yield enc, (torch.tensor([labels[i] for i in b], device=DEV) if labels is not None else None)


@torch.no_grad()   # no learning here, only predicting (saves memory and time)
def predict_probs(model, tok, texts):
    # Return, for every sentence, the probability of each of the six labels
    model.eval()
    out = [torch.softmax(model(**enc).logits.float(), -1).cpu().numpy() for enc, _ in batches(tok, texts, bs=32)]
    return np.concatenate(out)


def train_encoder(base, df, epochs=4, lr=3e-5, bs=16, seed=42):
    # Fine-tune any BERT-style model (FinBERT or DeBERTa) on our labelled sentences
    torch.manual_seed(seed); np.random.seed(seed)        # fixed seed = the same result every time
    texts, y = df.input.tolist(), to_label(df)
    tok = AutoTokenizer.from_pretrained(base)
    # Load the pre-trained model and give it a NEW final layer with six outputs (our six labels)
    model = AutoModelForSequenceClassification.from_pretrained(
        base, num_labels=len(LABELS), ignore_mismatched_sizes=True).to(DEV)

    # Class weights: rare labels count more in the loss. The square root softens this.
    counts = np.bincount(y, minlength=len(LABELS))
    w = torch.tensor((len(y) / (len(LABELS) * np.maximum(counts, 1))) ** 0.5, dtype=torch.float, device=DEV)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)        # measures how wrong the model's answers are

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)   # the "teacher" that adjusts the model
    steps = epochs * ((len(y) + bs - 1) // bs)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)    # warm up, then slowly lower the speed

    for ep in range(epochs):
        model.train(); total = 0
        for enc, lab in batches(tok, texts, y, bs=bs, shuffle=True, seed=seed + ep):
            loss = loss_fn(model(**enc).logits, lab)     # 1. how wrong were we on these 16 sentences?
            loss.backward()                              # 2. work out which direction makes us less wrong
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 3. don't jump too far in one go
            opt.step(); sched.step(); opt.zero_grad()    # 4. take the step, then reset for the next batch
            total += loss.item()
        # After every epoch, check progress on VALIDATION (never on test)
        pv = predict_probs(model, tok, val.input.tolist())
        print(f"epoch {ep + 1}: training loss {total:.1f} | validation F1 "
              f"{f1_score(val.is_risk, (1 - pv[:, 0] >= 0.5).astype(int)):.2f}")
    return model, tok


print("Device:", DEV, "| training will run:", RUN_TRAINING)

Device: cpu | training will run: False


In [10]:
if RUN_TRAINING:
    # Train FinBERT from scratch, exactly like src/models/finbert.py (takes 10-20 minutes on a laptop)
    fb_model, fb_tok = train_encoder("ProsusAI/finbert", train)
    P = predict_probs(fb_model, fb_tok, test.input.tolist())
    finbert_test_score = 1 - P[:, 0]                       # risk score = 1 - P(None)
else:
    # Load the FinBERT we already trained (saved in models/finbert) through our app's own code
    from src.app.pipeline import Classifier
    t0 = time.time()
    finbert = Classifier("finbert")
    finbert_test_score = finbert.predict(test.input.tolist())[0]
    print(f"scored {len(test)} test sentences in {time.time() - t0:.0f} s")

pred_finbert = (finbert_test_score >= 0.5).astype(int)
score("3. FinBERT, fine-tuned", test.is_risk, pred_finbert)

W1126 01:56:29.398000 54231 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   2%|▏         | 5/201 [00:00<00:09, 20.20it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 645.10it/s]

scored 600 test sentences in 25 s


{'model': '3. FinBERT, fine-tuned',
 'precision': 0.7,
 'recall': 0.84,
 'F1': 0.77}

**A big jump: 0.77.** FinBERT understands that *"there has been no default"* is not a risk, even though it contains the
word *default*. This is the model the app uses when it needs to be fast.

*(If you set `RUN_TRAINING = True`, your numbers may differ by a point or two. Training has some randomness, which is
exactly why we later average three copies.)*

## 7. Teaching the model what is *not* a risk: extra examples

Looking at the mistakes, the models often flagged two kinds of harmless text:

1. **Nil statements**: *"There were no instances of default in repayment of loans."* (says nothing bad happened)
2. **Template text**: standard paragraphs that appear in almost every report (auditor templates, legal boilerplate)

So we added **600 extra non-risk examples** (300 of each kind), taken **only from training companies' reports**,
never from validation or test companies. Script: `src/models/augment.py`.

We also tried a third kind, automatically inserting "not" into risk sentences, and **dropped it**: many rewritten
sentences were still risky, which would have taught the model wrong labels.

In [11]:
aug = pd.read_json(DATA / "train_aug.jsonl", lines=True)
print("extra non-risk examples:", len(aug), dict(aug.aug.value_counts()))
print("taken only from training companies:", set(aug.company) <= tr_c)
aug.groupby("aug").head(2)[["aug", "company", "text"]]

extra non-risk examples: 600 {'nil': np.int64(300), 'template': np.int64(300)}
taken only from training companies: True


,aug,company,text
0,nil,ULTRACEMCO,"Since there is no unanimous consent required from the parties, in the judgement of the management the Company does not have joint control over the..."
1,nil,COFFEEDAY,"DETAILS IN RESPECT OF FRAUDS REPORTED BY AUDITORS UNDER SECTION 143(12) There was no instance of fraud during the year under review, which require..."
300,template,LT,It is neither an audit nor an expression of opinion on the financial statements of the Company.
301,template,INFY,• Internal control systems and their adequacy


## 8. Step 4: a bigger model, DeBERTa-v3-large, trained three times and averaged

**DeBERTa-v3-large** is a newer and much bigger model than FinBERT (about 435 million parameters against 110 million).
It is better at understanding how words in a sentence relate to each other.

We trained it with the **same recipe** as FinBERT (the `train_encoder` function above), on the training sentences
**plus the 600 extra examples**, on a rented GPU. We trained it **3 times with different random seeds** (1, 2 and 42)
and **averaged their answers**. One copy can be lucky or unlucky; the average of three is more stable.

**How we chose the winner:** we compared 3 models (FinBERT, DeBERTa-base, DeBERTa-large) × with/without extra examples
× 3 seeds = 18 runs, and picked the best one **on the validation set only**. The test set was used once, at the end, to
report the score. That's how we avoid fooling ourselves.

Training DeBERTa-large needs a GPU, so here we load the saved answers of the three copies (`reports/probs/`).

In [12]:
# How it was trained (on a GPU), for reference -- the same function as above:
#   for seed in [1, 2, 42]:
#       model, tok = train_encoder("microsoft/deberta-v3-large", pd.concat([train, aug_as_non_risk]), seed=seed)

def saved_probs(seed):
    # The 600-sentence test = the original 300 sentences + 300 added later (v1.1), in that order
    P = ROOT / "reports/probs"
    return np.r_[np.load(P / f"debv3l_aug_s{seed}_test.npy"), np.load(P / f"debv3l_aug_s{seed}_testext.npy")]

copies = {seed: saved_probs(seed) for seed in [1, 2, 42]}

# Each copy on its own:
for seed, Pr in copies.items():
    f1 = f1_score(test.is_risk, (1 - Pr[:, 0] >= 0.5).astype(int))
    print(f"DeBERTa-large copy with seed {seed:>2}: F1 {f1:.2f}")

# The average of the three copies (the "ensemble"):
P_ens = np.mean(list(copies.values()), axis=0)
pred_deberta = (1 - P_ens[:, 0] >= 0.5).astype(int)
score("4. DeBERTa-v3-large + extra examples, 3 copies averaged", test.is_risk, pred_deberta)

DeBERTa-large copy with seed  1: F1 0.80
DeBERTa-large copy with seed  2: F1 0.84
DeBERTa-large copy with seed 42: F1 0.87


{'model': '4. DeBERTa-v3-large + extra examples, 3 copies averaged',
 'precision': 0.81,
 'recall': 0.92,
 'F1': 0.86}

**Our best model: F1 0.86, recall 0.92.** It finds 34 of the 37 real risks in the test set. Both targets we set at the
start (F1 ≥ 0.80, recall ≥ 0.85) are met.

The catch: on a laptop CPU, DeBERTa-large is slow. One copy takes about **10–16 minutes** for a whole report. That leads to
the last idea.

## 9. Step 5: the cascade, fast and accurate on a normal laptop

**Idea:** most sentences in a report are obviously harmless (*"The Board met 6 times this year"*). We don't need the
big model for those.

1. **FinBERT (fast)** scores every sentence.
2. Only sentences with a FinBERT score **≥ 0.05** go to **DeBERTa (accurate)**, which gives the final score.
3. All other sentences are treated as non-risk.

The bar of 0.05 is very low on purpose, so almost no real risk is thrown away at step 1. It isn't a number we tuned on
the test set: it is FinBERT's review threshold, chosen on validation.

Like a hospital: a quick check for everyone, the specialist only for the patients who need it.

In [13]:
from src.app.pipeline import Classifier

GATE = 0.05
passed = finbert_test_score >= GATE                               # which sentences go to DeBERTa?
print(f"FinBERT passes {passed.sum()} of {len(test)} sentences to DeBERTa ({100 * passed.mean():.0f}%)")
print("real risks thrown away at step 1:", int(((~passed) & (test.is_risk.values == 1)).sum()), "of", test.is_risk.sum())

t0 = time.time()
deberta = Classifier("deberta")                                   # the saved single DeBERTa-large + aug model
final_score = np.zeros(len(test))                                 # step 3: everything not passed = 0 (no risk)
final_score[passed] = deberta.predict(test.input[passed].tolist())[0]   # step 2: DeBERTa re-scores the passed ones
print(f"DeBERTa re-checked them in {time.time() - t0:.0f} s")

pred_cascade = (final_score >= 0.5).astype(int)
score("5. Cascade: FinBERT screens, DeBERTa re-checks", test.is_risk, pred_cascade)

FinBERT passes 70 of 600 sentences to DeBERTa (12%)
real risks thrown away at step 1: 3 of 37


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loading weights:   1%|▏         | 5/394 [00:02<03:03,  2.12it/s]

Loading weights: 100%|██████████| 394/394 [00:02<00:00, 165.50it/s]

DeBERTa re-checked them in 19 s


{'model': '5. Cascade: FinBERT screens, DeBERTa re-checks',
 'precision': 0.8,
 'recall': 0.86,
 'F1': 0.83}

**F1 0.83 at a fraction of the cost.** DeBERTa only reads about 12% of the sentences, so a whole report takes
about **2.5 minutes** on a laptop instead of 10–16. This is the setup we use in the demo (`REDFLAG_MODEL=cascade`).

## 10. Step 6: which flags are *serious*? The severity rule

A bank doesn't care equally about every flag. *"Raw material prices may rise"* is a risk, but *"the company defaulted
on its loans"* is an alarm. So on top of the model, we mark the most serious flags as **severe** with a transparent
rule, based on the **RBI's early-warning signals** that banks already use: going-concern doubt, loan default,
pledge invoked by lenders, covenant breach, NCLT / insolvency, fraud or forensic audit, rating downgrade,
statutory dues unpaid for over 6 months, and so on. Code: `src/models/rules.py`.

The rule also **removes false alarms**: standard auditor wording about "going concern" that appears in every report,
NCLT mentions that are only about a merger, and "nil" statements like "there was no default".

In [14]:
from rules import is_severe

examples = [
    "The Company has defaulted in repayment of loans to banks amounting to Rs 1,200 crore.",
    "The lenders have invoked the pledge on 2.5 crore equity shares held by the promoters.",
    "These events indicate that a material uncertainty exists that may cast significant doubt on the Company's ability to continue as a going concern.",
    # --- the ones below LOOK alarming but are harmless; the rule correctly says False ---
    "In preparing the financial statements, management is responsible for assessing the Company's ability to continue as a going concern.",
    "The Scheme of Amalgamation was sanctioned by the National Company Law Tribunal during the year.",
    "There has been no default in repayment of loans or borrowings to banks.",
]
pd.DataFrame({"sentence": examples, "severe?": [is_severe(s) for s in examples]})

,sentence,severe?
0,"The Company has defaulted in repayment of loans to banks amounting to Rs 1,200 crore.",True
1,The lenders have invoked the pledge on 2.5 crore equity shares held by the promoters.,True
2,These events indicate that a material uncertainty exists that may cast significant doubt on the Company's ability to continue as a going concern.,True
3,"In preparing the financial statements, management is responsible for assessing the Company's ability to continue as a going concern.",False
4,The Scheme of Amalgamation was sanctioned by the National Company Law Tribunal during the year.,False
5,There has been no default in repayment of loans or borrowings to banks.,False


## 11. Does it work on whole companies? The backtest

Sentence scores are nice, but a bank asks about **companies**. So we ran the full tool (cascade + severity rule) on
8 whole annual reports and computed one number per company:

> **severe flags per 1,000 sentences** (dividing by the report's length makes long and short reports comparable)

Then **ROC-AUC** answers: *if we pick one failed company and one healthy company at random, how often does the failed
one get the higher score?* 0.5 = guessing, 1.0 = perfect.

Here we recompute it from the saved outputs of each report, with the current severity rule.

In [15]:
groups = {   # what really happened to each company (1 = failed, 0 = healthy) and how we used it
    "GENSOL_FY2024": (1, "never used (collapsed 2025)"), "JETAIRWAYS_FY2018": (1, "test"), "YESBANK_FY2019": (1, "test"),
    "TCS_FY2019": (0, "test"), "MARUTI_FY2019": (0, "test"),
    "RCOM_FY2018": (1, "validation"), "TITAN_FY2019": (0, "validation"), "NTPC_FY2019": (0, "validation"),
}
rows = []
for name, (failed, used_as) in groups.items():
    r = json.loads((ROOT / f"reports/backtest/cascade/{name}.json").read_text())   # saved output of the full tool
    high = [f for f in r["findings"] if f["tier"] == "high"]                         # the confident flags
    severe = sum(is_severe(f["text"]) for f in high)                                 # how many are severe
    rows.append({"report": name, "failed?": failed, "used as": used_as, "sentences": r["sentences_analysed"],
                 "severe flags": severe, "severe per 1,000": round(1000 * severe / r["sentences_analysed"], 2)})
bt = pd.DataFrame(rows).sort_values("severe per 1,000", ascending=False)
bt

,report,failed?,used as,sentences,severe flags,"severe per 1,000"
5,RCOM_FY2018,1,validation,2853,36,12.62
1,JETAIRWAYS_FY2018,1,test,1954,9,4.61
2,YESBANK_FY2019,1,test,3470,6,1.73
6,TITAN_FY2019,0,validation,2730,3,1.10
0,GENSOL_FY2024,1,never used (collapsed 2025),2877,3,1.04
7,NTPC_FY2019,0,validation,3476,2,0.58
3,TCS_FY2019,0,test,2497,0,0.00
4,MARUTI_FY2019,0,test,2440,0,0.00


In [16]:
auc_all = roc_auc_score(bt["failed?"], bt["severe per 1,000"])
never_used = bt[bt["used as"].isin(["test", "never used (collapsed 2025)"])]   # the 5 reports no setting was chosen on
auc_unseen = roc_auc_score(never_used["failed?"], never_used["severe per 1,000"])

print(f"ROC-AUC on all 8 reports:          {auc_all:.2f}")
print(f"ROC-AUC on the 5 never-used ones:  {auc_unseen:.2f}  (every failed company above every healthy one)")

ROC-AUC on all 8 reports:          0.94
ROC-AUC on the 5 never-used ones:  1.00  (every failed company above every healthy one)


**How to read this:** on the 5 reports we never used to choose anything, every failed company (Jet Airways, Yes Bank,
Gensol) scores above every healthy one (TCS, Maruti): AUC **1.00**. On all 8 it is **0.94**: the one mix-up is Titan
(healthy), whose report discusses its exposure to the IL&FS collapse, and which scores slightly above Gensol.

*Note: Yes Bank shows 1.73 here; the backtest slide shows 2.02, computed with our first severe-sign list. The current
list (v2) is stricter about standard wording. The ranking and the AUC are the same.*

We later repeated this on **10 brand-new reports** (AUC 0.96) and a final clean exam on 4 reports nobody had
opened (AUC 1.00). See `src/eval/new_reports.py` and `reports/eval/severity_v2.json`.

## 12. Everything side by side

In [17]:
pd.DataFrame(results).set_index("model")

,precision,recall,F1
model,,,
1. Keyword search (Ctrl+F),0.13,0.73,0.23
2. TF-IDF + logistic regression,0.52,0.70,0.60
"3. FinBERT, fine-tuned",0.70,0.84,0.77
"4. DeBERTa-v3-large + extra examples, 3 copies averaged",0.81,0.92,0.86
"5. Cascade: FinBERT screens, DeBERTa re-checks",0.80,0.86,0.83


## 13. Viva cheat sheet: what each part proves

| If they ask… | Point to… | Say… |
|---|---|---|
| "How do you know there's no leakage?" | Section 2, the `assert` cell | "Each company is in one group only, and Gensol is in none. The code checks it." |
| "Why not just keywords?" | Section 4 | "Precision 0.13: keywords can't tell *no default* from *default*." |
| "What does fine-tuning mean?" | Section 6, `train_encoder` | "We start from FinBERT, which already knows finance, and train it 4 epochs on our 1,590 labelled sentences." |
| "Why six labels?" | Section 6 | "One model answers both questions: is it a risk, and what type. Risk score = 1 − P(None)." |
| "Why class weights?" | Section 6, `w = ...` | "Only 13% of sentences are risks, so we make them count more, softened with a square root." |
| "Why the extra examples?" | Section 7 | "The models kept flagging nil statements and template text, so we showed them 600 real examples of those." |
| "How did you choose the best model?" | Section 8 | "18 runs, picked on validation only. Test was used once, to report." |
| "Why average three copies?" | Section 8, the seed printout | "Single copies vary by a few points; the average is more stable." |
| "Why the cascade?" | Section 9 | "DeBERTa alone takes 10–16 min per report on a laptop. The cascade sends it only 12% of sentences: 2.5 min, F1 0.83." |
| "Why is 'severe' a rule, not a model?" | Section 10 | "It follows the RBI's official early-warning list, and a rule can explain exactly why something was flagged." |
| "Does it work on real companies?" | Section 11 | "On 5 never-used reports every failed company scores above every healthy one: AUC 1.00." |